In [4]:
import pandas as pd
from pathlib import Path

metrix_path = Path('metrix_jam.csv')
data_fix_path = Path('data_fix_sorted_terbaru.csv')

# Kedua file menggunakan delimiter ';'
df_metrix_jam = pd.read_csv(metrix_path, sep=';', usecols=[0, 1, 2])
df_data_fix = pd.read_csv(data_fix_path, sep=';')

print('df_metrix_jam shape:', df_metrix_jam.shape)
print('df_data_fix shape  :', df_data_fix.shape)

display(df_metrix_jam.head(5))
display(df_data_fix.head(5))

df_metrix_jam shape: (999, 3)
df_data_fix shape  : (3012, 73)


,Judul,Tanggal,Jam Upload (WIB)
0,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR...,2025-12-01,7.3
1,"INDONESIA STOP IMPOR BERAS, JAGUNG, DAN GARAM‼...",2025-12-01,12.3
2,INDONESIA KALAHKAN ISRAEL & IRAN MILITER...,2025-12-01,15.3
3,TNI AL KEPUNG AMBALAT‼... (Shorts),2025-12-01,17.0
4,TAIWAN GELONTORKAN RP 666 T BIKIN T-DOME...,2025-12-01,18.3


,Konten,Judul video,Waktu publikasi video,Durasi,Penayangan tak dilewati,Rata-rata durasi tonton,Persentase penayangan rata-rata (%),Tetap menonton (%),Penonton unik,Penayangan rata-rata per penonton,...,Klik teaser per teaser kartu yang ditampilkan (%),Klik pada elemen layar akhir,Klik per elemen layar akhir yang ditampilkan (%),Elemen layar akhir yang ditampilkan,Penayangan,Waktu tonton (jam),Subscriber,Estimasi pendapatan (IDR),Tayangan,Rasio klik-tayang dari tayangan (%)
0,OSDYAzU5v-I,"AUSTRALIA & INGGRIS PERINGATI HUT OPM, PAPUA S...",2024-12-01,505.0,16938.0,0:02:39,31.57,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,16938.0,750.0687,61.0,56004.112,108374.0,13.03
1,OSDYAzU5v-I,"AUSTRALIA & INGGRIS PERINGATI HUT OPM, PAPUA S...",2024-12-01,505.0,16938.0,0:02:39,31.57,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,16938.0,750.0687,61.0,56004.112,108374.0,13.03
2,OSDYAzU5v-I,"AUSTRALIA & INGGRIS PERINGATI HUT OPM, PAPUA S...",2024-12-01,505.0,36169.0,0:02:44,32.53,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,36169.0,1650.3845,125.0,101546.243,221456.0,13.61
3,OSDYAzU5v-I,"AUSTRALIA & INGGRIS PERINGATI HUT OPM, PAPUA S...",2024-12-01,505.0,36169.0,0:02:44,32.53,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,36169.0,1650.3845,125.0,101546.243,221456.0,13.61
4,OSDYAzU5v-I,PAPUA MASUK DALAM DAFTAR NEGARA-NEGARA BARU YA...,2024-12-04,534.0,11987.0,0:02:20,26.31,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,11987.0,467.7960,60.0,36525.391,67377.0,13.80


In [6]:
import re

# 1) Samakan format tanggal
left = df_metrix_jam.copy()
right = df_data_fix.copy()

left['Tanggal'] = pd.to_datetime(left['Tanggal'], errors='coerce').dt.date
right['Waktu publikasi video'] = pd.to_datetime(right['Waktu publikasi video'], errors='coerce').dt.date

# 2) Ambil 4 kata pertama dari judul setelah dibersihkan
#    (lowercase, buang simbol, rapikan spasi)
def first_n_words(text, n=4):
    if pd.isna(text):
        return ''
    cleaned = str(text).lower()
    cleaned = re.sub(r'[^a-z0-9\s]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return ' '.join(cleaned.split()[:n])

left['judul_4kata'] = left['Judul'].apply(first_n_words)
right['judul_4kata'] = right['Judul video'].apply(first_n_words)

# 3) Buat key gabungan: tanggal + 4 kata pertama
left['merge_key'] = left['Tanggal'].astype(str) + '|' + left['judul_4kata']
right['merge_key'] = right['Waktu publikasi video'].astype(str) + '|' + right['judul_4kata']

# 4) Merge berdasarkan key (many-to-many jika ada key duplikat)
merged = left.merge(
    right,
    on='merge_key',
    how='inner',
    suffixes=('_metrix', '_datafix')
)

# Diagnostik duplikasi key
dup_left = left['merge_key'].duplicated().sum()
dup_right = right['merge_key'].duplicated().sum()

print('Jumlah baris metrix_jam :', len(left))
print('Jumlah baris data_fix   :', len(right))
print('Duplikat key kiri       :', dup_left)
print('Duplikat key kanan      :', dup_right)
print('Jumlah baris match      :', len(merged))

# Versi 1-1 per key (opsional): ambil baris pertama tiap key dari masing-masing tabel
left_u = left.drop_duplicates(subset=['merge_key'], keep='first')
right_u = right.drop_duplicates(subset=['merge_key'], keep='first')
merged_unique = left_u.merge(
    right_u,
    on='merge_key',
    how='inner',
    suffixes=('_metrix', '_datafix')
)
print('Jumlah match unik key   :', len(merged_unique))

# Tampilkan kolom penting agar mudah cek hasil
cols_preview = [
    'Judul', 'Tanggal', 'Jam Upload (WIB)',
    'Judul video', 'Waktu publikasi video',
    'merge_key'
]
display(merged[cols_preview].head(20))
display(merged_unique[cols_preview].head(20))

Jumlah baris metrix_jam : 999
Jumlah baris data_fix   : 3012
Duplikat key kiri       : 237
Duplikat key kanan      : 1847
Jumlah baris match      : 7750
Jumlah match unik key   : 524


,Judul,Tanggal,Jam Upload (WIB),Judul video,Waktu publikasi video,merge_key
0,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR...,2025-12-01,7.3,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR LI...,2025-12-01,2025-12-01|tni al kepung ambalat
1,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR...,2025-12-01,7.3,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR LI...,2025-12-01,2025-12-01|tni al kepung ambalat
2,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR...,2025-12-01,7.3,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR LI...,2025-12-01,2025-12-01|tni al kepung ambalat
3,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR...,2025-12-01,7.3,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR LI...,2025-12-01,2025-12-01|tni al kepung ambalat
4,INDONESIA KALAHKAN ISRAEL & IRAN MILITER...,2025-12-01,15.3,INDONESIA KALAHKAN ISRAEL & IRAN MILITER TERKU...,2025-12-01,2025-12-01|indonesia kalahkan israel iran
5,INDONESIA KALAHKAN ISRAEL & IRAN MILITER...,2025-12-01,15.3,INDONESIA KALAHKAN ISRAEL & IRAN MILITER TERKU...,2025-12-01,2025-12-01|indonesia kalahkan israel iran
6,INDONESIA KALAHKAN ISRAEL & IRAN MILITER...,2025-12-01,15.3,INDONESIA KALAHKAN ISRAEL & IRAN MILITER TERKU...,2025-12-01,2025-12-01|indonesia kalahkan israel iran
7,INDONESIA KALAHKAN ISRAEL & IRAN MILITER...,2025-12-01,15.3,INDONESIA KALAHKAN ISRAEL & IRAN MILITER TERKU...,2025-12-01,2025-12-01|indonesia kalahkan israel iran
8,TNI AL KEPUNG AMBALAT‼... (Shorts),2025-12-01,17.0,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR LI...,2025-12-01,2025-12-01|tni al kepung ambalat
9,TNI AL KEPUNG AMBALAT‼... (Shorts),2025-12-01,17.0,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR LI...,2025-12-01,2025-12-01|tni al kepung ambalat


,Judul,Tanggal,Jam Upload (WIB),Judul video,Waktu publikasi video,merge_key
0,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR...,2025-12-01,7.3,TNI AL KEPUNG AMBALAT‼ MALAYSIA KETAR-KETIR LI...,2025-12-01,2025-12-01|tni al kepung ambalat
1,INDONESIA KALAHKAN ISRAEL & IRAN MILITER...,2025-12-01,15.3,INDONESIA KALAHKAN ISRAEL & IRAN MILITER TERKU...,2025-12-01,2025-12-01|indonesia kalahkan israel iran
2,"PAPUA BARAT JADI REBUTAN ASING, PRABOWO...",2025-12-02,7.3,"PAPUA BARAT JADI REBUTAN ASING, PRABOWO NGAMUK...",2025-12-02,2025-12-02|papua barat jadi rebutan
3,SUDAN–ANGOLA–ETHIOPIA GABUNG INDONESIA...,2025-12-02,9.3,"SUDAN–ANGOLA–ETHIOPIA GABUNG INDONESIA, MALAYS...",2025-12-02,2025-12-02|sudan angola ethiopia gabung
4,"MALAYSIA DIBUAT CEMAS, INDONESIA & AUSTRALIA...",2025-12-02,12.3,"MALAYSIA DIBUAT CEMAS, INDONESIA & AUSTRALIA B...",2025-12-02,2025-12-02|malaysia dibuat cemas indonesia
5,"INDONESIA LOLOS JEBAKAN AS, MALAYSIA MALAH...",2025-12-02,15.3,"INDONESIA LOLOS JEBAKAN AS, MALAYSIA MALAH TER...",2025-12-02,2025-12-02|indonesia lolos jebakan as
6,"MALAYSIA MAKIN CIUT NYALINYA, INDONESIA GANDEN...",2025-12-03,7.3,"MALAYSIA MAKIN CIUT NYALINYA, INDONESIA GANDEN...",2025-12-03,2025-12-03|malaysia makin ciut nyalinya
7,INDIA & JEPANG BANGUN POROS BARU DENGAN...,2025-12-03,10.3,INDIA & JEPANG BANGUN POROS BARU DENGAN INDONE...,2025-12-03,2025-12-03|india jepang bangun poros
8,"IRAN UNDANG INDONESIA, PRABOWO PENENTU...",2025-12-03,12.3,"IRAN UNDANG INDONESIA, PRABOWO PENENTU PERDAMA...",2025-12-03,2025-12-03|iran undang indonesia prabowo
9,INDONESIA KENDALIKAN SUPLAI EMAS DUNIA...,2025-12-03,15.3,"INDONESIA KENDALIKAN SUPLAI EMAS DUNIA, AMERIK...",2025-12-03,2025-12-03|indonesia kendalikan suplai emas


In [7]:
# Export hasil merge untuk analisis
output_unique = 'merged_unique_4kata_tanggal.csv'
output_all = 'merged_all_4kata_tanggal.csv'

merged_unique.to_csv(output_unique, index=False)
merged.to_csv(output_all, index=False)

print(f'Berhasil export: {output_unique} ({len(merged_unique)} baris)')
print(f'Berhasil export: {output_all} ({len(merged)} baris)')

Berhasil export: merged_unique_4kata_tanggal.csv (524 baris)
Berhasil export: merged_all_4kata_tanggal.csv (7750 baris)
